In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded successfully!")

In [ ]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch

# -----------------------------
# Config
# -----------------------------
VG_ROOT = Path("../data/visual_genome")
VG_OBJECTS_PATH = VG_ROOT / "VisualGenome_task" / "objects.json"
OUTPUT_PATH = "../results/infer/visual_genome/llava15/dola_low.json"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 256
PROMPT = "Describe this image."

# DoLa config
DOLA_LAYERS = "low"
REPETITION_PENALTY = 1.2

# tokenizer / generation setup
processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id = processor.tokenizer.eos_token_id

In [ ]:
def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def resolve_vg_image_path(vg_root: Path, image_id: int) -> Path:
    """
    Supports full Visual Genome structure:
      data/visual_genome/images/VG_100K/{id}.jpg
      data/visual_genome/images2/VG_100K_2/{id}.jpg
    """
    candidates = [
        vg_root / "images2" / "VG_100K_2" / f"{image_id}.jpg",
        vg_root / "images" / "VG_100K" / f"{image_id}.jpg",
        vg_root / f"{image_id}.jpg",  # fallback
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find image {image_id}.jpg. Tried:\n" +
        "\n".join(str(p) for p in candidates)
    )


def load_image(path: Path):
    return Image.open(path).convert("RGB")


def clean_output(text: str) -> str:
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text

In [ ]:
# -----------------------------
# Load first 100 Visual Genome image IDs
# -----------------------------
with open(VG_OBJECTS_PATH, "r", encoding="utf-8") as f:
    vg_objects = json.load(f)

vg_objects = vg_objects[:100]
image_ids = [int(obj["image_id"]) for obj in vg_objects]

print(f"Loaded {len(image_ids)} Visual Genome image IDs.")
print("First 5 IDs:", image_ids[:5])

In [ ]:
# -----------------------------
# Load first 100 Visual Genome image IDs
# -----------------------------
with open(VG_OBJECTS_PATH, "r", encoding="utf-8") as f:
    vg_objects = json.load(f)

vg_objects = vg_objects[:100]
image_ids = [int(obj["image_id"]) for obj in vg_objects]

print(f"Loaded {len(image_ids)} Visual Genome image IDs.")
print("First 5 IDs:", image_ids[:5])

In [ ]:
# -----------------------------
# Run batched DoLa-low inference
# -----------------------------
results = []
hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

num_batches = (len(image_ids) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx, batch_ids in enumerate(tqdm(batch_list(image_ids, BATCH_SIZE), total=num_batches)):
    batch_images = []
    batch_prompts = []
    batch_paths = []
    valid_ids = []

    # prepare one mini-batch
    for image_id in batch_ids:
        image_path = resolve_vg_image_path(VG_ROOT, image_id)
        image = load_image(image_path)

        batch_images.append(image)
        batch_prompts.append(hf_prompt)
        batch_paths.append(str(image_path))
        valid_ids.append(image_id)

    inputs = processor(
        text=batch_prompts,
        images=batch_images,
        return_tensors="pt",
        padding=True
    )

    if batch_idx == 0:
        print("input_ids batch shape:", inputs["input_ids"].shape)
        print("num images:", len(batch_images))
        print("num prompts:", len(batch_prompts))

    input_device = next(model.parameters()).device
    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,

            # DoLa-low
            dola_layers=DOLA_LAYERS,
            repetition_penalty=REPETITION_PENALTY,
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    for image_id, image_path, output in zip(valid_ids, batch_paths, outputs):
        caption = clean_output(output)

        results.append({
            "image_id": image_id,
            "path": image_path,
            "text": caption
        })